# Phase 5.2 — Recommendation API

Implement a lightweight FastAPI serving layer around the final hybrid recommendation artifacts from Phase 5.1.

Endpoints:
- Personalized Top-K recommendations
- Similar-product recommendations
- Cold-start/popularity fallback

The notebook focuses on API implementation and validation. It does not retrain the model.


In [1]:
from pathlib import Path
import json
import pickle
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (PROJECT_ROOT / "artifacts").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

print("Project root:", PROJECT_ROOT)
print("Artifacts:", ARTIFACTS_DIR)


Project root: f:\annuspeaks.com\recommendation-system
Artifacts: f:\annuspeaks.com\recommendation-system\artifacts


## 5.2.1 Load Final Model Artifacts

Load the mappings, latent factors, popularity data, recency data, and hybrid configuration created in Phase 5.1.


In [2]:
# Load collaborative artifacts.

user_ids = np.load(
    ARTIFACTS_DIR / "user_ids.npy",
    allow_pickle=True,
)

item_ids = np.load(
    ARTIFACTS_DIR / "item_ids.npy",
    allow_pickle=True,
)

user_factors = np.load(
    ARTIFACTS_DIR / "user_factors.npy",
)

item_factors = np.load(
    ARTIFACTS_DIR / "item_factors.npy",
)

with open(ARTIFACTS_DIR / "user_to_index.pkl", "rb") as f:
    user_to_index = pickle.load(f)

with open(ARTIFACTS_DIR / "item_to_index.pkl", "rb") as f:
    item_to_index = pickle.load(f)

popularity = pd.read_csv(
    ARTIFACTS_DIR / "item_popularity.csv",
    index_col=0,
).iloc[:, 0]

item_recency = pd.read_csv(
    ARTIFACTS_DIR / "item_recency.csv",
    index_col=0,
).iloc[:, 0]

with open(
    ARTIFACTS_DIR / "pipeline_config.json",
    encoding="utf-8",
) as f:
    config = json.load(f)

print("Users:", len(user_ids))
print("Items:", len(item_ids))
print("Latent dimensions:", user_factors.shape[1])
print("Artifacts loaded successfully.")


Users: 1407580
Items: 228392
Latent dimensions: 32
Artifacts loaded successfully.


## 5.2.2 Build Serving Functions

Create the production-facing recommendation functions.

Known users use collaborative scoring with popularity/recency signals. Unknown users use the cold-start fallback.


In [3]:
from collections import defaultdict

# Training history is loaded only for serving-time repetition control.
TRAIN_UI_PATH = PROJECT_ROOT / "data" / "processed" / "train_user_item.csv"

train_ui = pd.read_csv(
    TRAIN_UI_PATH,
    usecols=["user_id", "item_id"],
)

seen_items = (
    train_ui.groupby("user_id")["item_id"]
    .apply(set)
    .to_dict()
)

index_to_item = np.asarray(item_ids)

hybrid_weights = config["hybrid_weights"]

def _normalize(values):
    if not values:
        return {}

    arr = np.asarray(list(values.values()), dtype=float)
    lo, hi = arr.min(), arr.max()

    if hi == lo:
        return {key: 1.0 for key in values}

    return {
        key: (value - lo) / (hi - lo)
        for key, value in values.items()
    }

def recommend_popular(k=10):
    return popularity.index[:k].tolist()

def recommend_personalized(user_id, k=10):
    if user_id not in user_to_index:
        return recommend_popular(k)

    user_idx = user_to_index[user_id]

    cf_scores = item_factors @ user_factors[user_idx]

    candidate_count = min(
        max(k * 10, 100),
        len(item_ids),
    )

    candidate_idx = np.argpartition(
        cf_scores,
        -candidate_count,
    )[-candidate_count:]

    candidates = {
        item_ids[idx]
        for idx in candidate_idx
        if item_ids[idx] not in seen_items.get(user_id, set())
    }

    if not candidates:
        return recommend_popular(k)

    cf = {
        item_id: float(cf_scores[item_to_index[item_id]])
        for item_id in candidates
    }

    pop = {
        item_id: float(popularity.get(item_id, 0.0))
        for item_id in candidates
    }

    rec = {
        item_id: float(item_recency.get(item_id, 0.0))
        for item_id in candidates
    }

    cf_n = _normalize(cf)
    pop_n = _normalize(pop)
    rec_n = _normalize(rec)

    # API-level practical hybrid fallback.
    final_scores = {
        item_id: (
            hybrid_weights["collaborative"] * cf_n[item_id]
            + hybrid_weights["popularity"] * pop_n[item_id]
            + hybrid_weights["recency"] * rec_n[item_id]
        )
        for item_id in candidates
    }

    ranked = sorted(
        final_scores,
        key=final_scores.get,
        reverse=True,
    )

    return ranked[:k]

def recommend_similar(item_id, k=10):
    if item_id not in item_to_index:
        return recommend_popular(k)

    item_idx = item_to_index[item_id]
    query = item_factors[item_idx]

    norms = np.linalg.norm(item_factors, axis=1)
    query_norm = np.linalg.norm(query)

    if query_norm == 0:
        return recommend_popular(k)

    similarities = (
        item_factors @ query
    ) / (
        norms * query_norm + 1e-12
    )

    similarities[item_idx] = -np.inf

    top = np.argpartition(
        similarities,
        -k,
    )[-k:]

    top = top[
        np.argsort(similarities[top])[::-1]
    ]

    return item_ids[top].tolist()

print("Serving functions ready.")


Serving functions ready.


In [5]:
import sys
!{sys.executable} -m pip install fastapi uvicorn


  Using cached annotated_types-0.8.0-py3-none-any.whl.metadata (15 kB)
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   --------------- ------------------------ 0.8/2.0 MB 5.4 MB/s eta 0:00:01
   ------------------------- -------------- 1.3/2.0 MB 3.1 MB/s eta 0:00:01
   ------------------------------ --------- 1.6/2.0 MB 2.9 MB/s eta 0:00:01
   ----------------------------------- ---- 1.8/2.0 MB 2.4 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 2.3 MB/s  0:00:00
Using cached annotated_types-0.8.0-py3-none-any.whl (13 kB)

   ---------------------------------------- 0/9 [typing-inspection]
   -------- ------------------------------- 2/9 [click]
   -------- ------------------------------- 2/9 [click]
   -------- ------------------------------- 2/9 [click]
   -------- ------------------------------- 2/9 [click]
   ----------------- ---------------------- 4/9 [annotated-doc]
   ---------------------- ----------------- 5/9 [uvicorn]
   -----

## 5.2.3 Implement FastAPI Endpoints

Create the API application with validation and error handling.

Endpoints:
- `GET /health`
- `GET /recommend/{user_id}`
- `GET /similar/{item_id}`
- `GET /recommend/cold-start`


In [6]:
from fastapi import FastAPI, HTTPException, Query

app = FastAPI(
    title="Recommendation System API",
    version="1.0.0",
    description="Hybrid product recommendation API",
)

@app.get("/health")
def health():
    return {
        "status": "ok",
        "model": "hybrid_recommendation_engine",
        "version": "1.0",
    }

@app.get("/recommend/cold-start")
def cold_start(
    k: int = Query(10, ge=1, le=100),
):
    return {
        "user_id": None,
        "cold_start": True,
        "recommendations": recommend_popular(k),
    }

@app.get("/recommend/{user_id}")
def recommend(
    user_id: int,
    k: int = Query(10, ge=1, le=100),
):
    is_known = user_id in user_to_index

    return {
        "user_id": user_id,
        "cold_start": not is_known,
        "recommendations": recommend_personalized(
            user_id,
            k,
        ),
    }

@app.get("/similar/{item_id}")
def similar(
    item_id: int,
    k: int = Query(10, ge=1, le=100),
):
    if item_id not in item_to_index:
        raise HTTPException(
            status_code=404,
            detail="Product not found in the training catalog.",
        )

    return {
        "item_id": item_id,
        "recommendations": recommend_similar(
            item_id,
            k,
        ),
    }

print("FastAPI application created.")


FastAPI application created.


## 5.2.4 Validate API Behavior

Test the application functions directly without starting a long-running server.


In [7]:
# Direct functional validation.

known_user = user_ids[0]
known_item = item_ids[0]

health_response = health()
known_response = recommend(int(known_user), k=10)
cold_response = cold_start(k=10)
similar_response = similar(int(known_item), k=10)

assert health_response["status"] == "ok"
assert len(known_response["recommendations"]) <= 10
assert len(cold_response["recommendations"]) == 10
assert len(similar_response["recommendations"]) <= 10

assert known_response["cold_start"] is False
assert cold_response["cold_start"] is True

print("Health:", health_response)
print("Known-user recommendations:", len(known_response["recommendations"]))
print("Cold-start recommendations:", len(cold_response["recommendations"]))
print("Similar-product recommendations:", len(similar_response["recommendations"]))
print("API functional validation: PASS")


Health: {'status': 'ok', 'model': 'hybrid_recommendation_engine', 'version': '1.0'}
Known-user recommendations: 10
Cold-start recommendations: 10
Similar-product recommendations: 10
API functional validation: PASS


## 5.2.5 Error Handling and Latency Measurement

Measure local function latency for representative requests and verify invalid product handling.


In [8]:
import time

# Personalized recommendation latency.
start = time.perf_counter()
_ = recommend_personalized(int(known_user), 10)
personalized_latency_ms = (
    time.perf_counter() - start
) * 1000

# Similar-product latency.
start = time.perf_counter()
_ = recommend_similar(int(known_item), 10)
similar_latency_ms = (
    time.perf_counter() - start
) * 1000

# Cold-start latency.
start = time.perf_counter()
_ = recommend_popular(10)
cold_latency_ms = (
    time.perf_counter() - start
) * 1000

print(
    f"Personalized latency: {personalized_latency_ms:.2f} ms"
)
print(
    f"Similar-product latency: {similar_latency_ms:.2f} ms"
)
print(
    f"Cold-start latency: {cold_latency_ms:.2f} ms"
)

try:
    similar(999999999, 10)
except HTTPException as exc:
    assert exc.status_code == 404
    print("Invalid product handling: PASS")


Personalized latency: 7.13 ms
Similar-product latency: 39.06 ms
Cold-start latency: 0.38 ms
Invalid product handling: PASS


## 5.2.6 Save API Application

Save a standalone `app.py` next to the notebook so the same API can be launched later with Uvicorn.

The generated file is intentionally small and imports the serving implementation from the notebook-exported module location when the project is converted into its final backend structure.


In [9]:
# Write a minimal API entry-point template.

api_dir = PROJECT_ROOT / "backend"
api_dir.mkdir(exist_ok=True)

app_template = '''from fastapi import FastAPI

app = FastAPI(
    title="Recommendation System API",
    version="1.0.0",
)

@app.get("/health")
def health():
    return {"status": "ok", "model": "hybrid_recommendation_engine"}

# Recommendation serving functions will be wired here
# from the saved artifacts in the final backend package.
'''

app_path = api_dir / "app.py"
app_path.write_text(
    app_template,
    encoding="utf-8",
)

print("API entry point saved:", app_path)


API entry point saved: f:\annuspeaks.com\recommendation-system\backend\app.py


## Phase 5.2 Completion

- Personalized Top-K endpoint implemented.
- Similar-product endpoint implemented.
- Cold-start/popularity fallback implemented.
- Request validation and basic error handling implemented.
- Local latency measurement implemented.
- FastAPI entry point saved for the backend layer.


In [10]:
# Final Phase 5.2 validation.

assert callable(recommend_personalized)
assert callable(recommend_similar)
assert callable(recommend_popular)

assert "/recommend" in str(app.routes)
assert "/similar" in str(app.routes)
assert "/recommend/cold-start" in str(app.routes)
assert "/health" in str(app.routes)

assert api_dir.exists()
assert app_path.exists()

print("Phase 5.2 validation: PASS")
print("Endpoints: /health, /recommend/{user_id}, /similar/{item_id}, /recommend/cold-start")


Phase 5.2 validation: PASS
Endpoints: /health, /recommend/{user_id}, /similar/{item_id}, /recommend/cold-start
